# IN4640 Assignment 2 — Question 2: Camera Model & Earring Size

**Name:** *Gayashan W.J*  
**Index:** *215525P*  

---

## Question
> Fig. 1 shows an image of a pair of earrings. Assume that a camera is mounted with the optical axis **perpendicular** to the imaging plane. If the **focal length** is 8 mm, the **pixel size** is 2.2 µm × 2.2 µm, and the **distance from the lens to the imaging plane** is 720 mm, what are the sizes of the earrings?

---

## Theory: Pinhole / Thin-Lens Camera Model

The **thin-lens equation** relates object distance $d_o$, image distance $d_i$, and focal length $f$:

$$\frac{1}{f} = \frac{1}{d_o} + \frac{1}{d_i}$$

Solving for object distance:

$$d_o = \frac{f \cdot d_i}{d_i - f}$$

The **lateral magnification** is:

$$m = \frac{d_i}{d_o}$$

The physical size of an object imaged onto the sensor is:

$$\text{sensor\_size} = N_{px} \times p$$

where $N_{px}$ is the number of pixels and $p$ is the pixel pitch (size).

The **real-world object size** is therefore:

$$\text{real\_size} = \frac{\text{sensor\_size}}{m} = \frac{N_{px} \times p \times d_o}{d_i}$$


In [8]:
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ── Load earring image ─────────────────────────────────────────────────────────
img     = cv2.imread("earrings.jpg")
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
H, W    = img.shape[:2]
print(f"Image resolution: {W} × {H} pixels")


Image resolution: 1024 × 1024 pixels


---
## Step 1: Measure Earring Diameter in Pixels

We segment the earrings from the white background using thresholding and find bounding boxes via contour detection.


In [9]:
# ── Segment earrings and measure diameter in pixels ───────────────────────────
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Invert threshold: earrings (gold) are darker than white background
_, thresh = cv2.threshold(gray, 230, 255, cv2.THRESH_BINARY_INV)

contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Two largest contours = two earrings
contours     = sorted(contours, key=cv2.contourArea, reverse=True)[:2]
bboxes       = [cv2.boundingRect(c) for c in contours]
diameters_px = [(bb[2] + bb[3]) / 2 for bb in bboxes]   # avg of width & height
avg_diam_px  = np.mean(diameters_px)

print("Bounding boxes (x, y, w, h):")
for i, bb in enumerate(bboxes):
    print(f"  Earring {i+1}: x={bb[0]}, y={bb[1]}, w={bb[2]}, h={bb[3]}")

print(f"\nDiameter estimates: {[f'{d:.1f} px' for d in diameters_px]}")
print(f"Average diameter:   {avg_diam_px:.1f} pixels")


Bounding boxes (x, y, w, h):
  Earring 1: x=101, y=298, w=381, h=399
  Earring 2: x=541, y=298, w=380, h=399

Diameter estimates: ['390.0 px', '389.5 px']
Average diameter:   389.8 pixels


In [10]:
# ── Visualise detection ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(img_rgb)

for i, (bb, d) in enumerate(zip(bboxes, diameters_px)):
    x, y, bw, bh = bb
    rect = patches.Rectangle((x, y), bw, bh, linewidth=2,
                              edgecolor='red', facecolor='none')
    ax.add_patch(rect)
    ax.text(x, y - 12, f'Earring {i+1}: {d:.0f} px',
            color='red', fontsize=11, fontweight='bold',
            bbox=dict(facecolor='white', alpha=0.7, pad=2))

ax.set_title(f'Earring Detection — avg diameter = {avg_diam_px:.0f} px',
             fontsize=12, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('q2_detection.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Step 2: Apply Camera Model


In [11]:
# ── Camera parameters ──────────────────────────────────────────────────────────
f_mm     = 8.0           # focal length [mm]
pixel_um = 2.2           # pixel pitch [µm]
pixel_mm = pixel_um*1e-3 # convert to mm = 0.0022 mm
di       = 720.0         # image distance: lens → sensor [mm]

print(f"Given parameters:")
print(f"  Focal length f       = {f_mm} mm")
print(f"  Pixel size           = {pixel_um} µm = {pixel_mm} mm")
print(f"  Image distance dᵢ    = {di} mm")


Given parameters:
  Focal length f       = 8.0 mm
  Pixel size           = 2.2 µm = 0.0022 mm
  Image distance dᵢ    = 720.0 mm


In [12]:
# ── Step 2a: Object distance via thin-lens equation ───────────────────────────
# 1/f = 1/do + 1/di  =>  do = f*di / (di - f)
do = f_mm * di / (di - f_mm)
print(f"Object distance  dₒ = f·dᵢ/(dᵢ-f) = {f_mm}×{di}/({di}-{f_mm})")
print(f"                    = {do:.4f} mm")

# ── Step 2b: Lateral magnification ────────────────────────────────────────────
m = di / do
print(f"\nMagnification    m  = dᵢ/dₒ = {di}/{do:.4f} = {m:.4f}×")

# ── Step 2c: Physical size on sensor ──────────────────────────────────────────
sensor_mm = avg_diam_px * pixel_mm
print(f"\nSensor image size = {avg_diam_px:.1f} px × {pixel_mm} mm/px = {sensor_mm:.6f} mm")

# ── Step 2d: Real-world size ───────────────────────────────────────────────────
real_mm = sensor_mm / m
print(f"\nReal-world size   = sensor_size / m = {sensor_mm:.6f} / {m:.4f}")
print(f"                  = {real_mm:.6f} mm")
print(f"                  = {real_mm*1000:.4f} µm")

print(f"\n{'='*50}")
print(f"RESULT: Earring outer diameter ≈ {real_mm:.6f} mm ({real_mm*1000:.4f} µm)")
print(f"{'='*50}")


Object distance  dₒ = f·dᵢ/(dᵢ-f) = 8.0×720.0/(720.0-8.0)
                    = 8.0899 mm

Magnification    m  = dᵢ/dₒ = 720.0/8.0899 = 89.0000×

Sensor image size = 389.8 px × 0.0022 mm/px = 0.857450 mm

Real-world size   = sensor_size / m = 0.857450 / 89.0000
                  = 0.009634 mm
                  = 9.6343 µm

RESULT: Earring outer diameter ≈ 0.009634 mm (9.6343 µm)


In [13]:
# ── Summary visualisation ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.axis('off')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

rows = [
    ("", "Parameter", "Value", True),
    ("Given", "Focal length  f", f"{f_mm} mm", False),
    ("Given", "Pixel size  p", f"{pixel_um} µm × {pixel_um} µm", False),
    ("Given", "Image distance  dᵢ", f"{di} mm", False),
    ("Given", "Measured diameter", f"{avg_diam_px:.0f} px", False),
    ("Computed", "Object distance  dₒ = f·dᵢ/(dᵢ−f)", f"{do:.4f} mm", False),
    ("Computed", "Magnification  m = dᵢ/dₒ", f"{m:.4f}×", False),
    ("Computed", "Sensor image size  = px × p", f"{sensor_mm:.6f} mm", False),
    ("RESULT", "Real-world earring diameter", f"{real_mm:.6f} mm  ≈  {real_mm*1000:.4f} µm", True),
]
table = ax.table(
    cellText=[[r[0], r[1], r[2]] for r in rows[1:]],
    colLabels=["Stage", "Formula / Parameter", "Value"],
    loc='center', cellLoc='left'
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#1a237e'); cell.set_text_props(color='white', fontweight='bold')
    elif rows[row][3]:
        cell.set_facecolor('#fff9c4')
    elif rows[row][0] == 'Computed':
        cell.set_facecolor('#e8f5e9')

ax.set_title('Q2: Earring Size Calculation Summary', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('q2_summary_table.png', dpi=150, bbox_inches='tight')
plt.show()
